# R-Script: Psychometric Evaluation of the German PPBS

This JupyterLab notebook contains the R code used for the psychometric evaluation of the German 5-item Pre- and Postnatal Bonding Scale (PPBS). 
Short explanatory notes are provided throughout the notebook to facilitate transparency and reproducibility of the analyses.

**Input objects required before running the analyses:**

- `data`: Cleaned WELCOME trial dataset containing the German PPBS and external variables.
- `original_data`: Original Dutch PPBS validation dataset, required only for the cross-version comparison.

The original trial data are not included in this repository due to ethical and data protection restrictions. The notebook imports the required files in the setup section. File paths refer to a local, non-versioned `data_raw/` folder and should be adapted as needed before running the code elsewhere.

# 1. Setup

## 1.1 Packages

Load packages used in the analyses.
Install missing packages locally before running the notebook.

In [ ]:
library(dplyr)        # Required for data manipulation, filtering, recoding, grouping, and summarising
library(ggplot2)      # Required for creating plots and visualisations
library(haven)        # Required for importing SPSS .sav files, including the original Dutch PPBS dataset
library(IRdisplay)    # Required for displaying HTML tables in Jupyter notebook outputs
library(knitr)        # Required for formatting analysis output tables using kable()
library(lavaan)       # Required for confirmatory factor analysis (CFA) and measurement invariance analyses
library(lmtest)       # Required for regression diagnostics
library(psych)        # Required for psychometric analyses
library(purrr)        # Required for functional programming helpers, e.g. mapping over PPBS time points
library(RColorBrewer) # Required for colour palettes used in visualisations
library(semTools)     # Required for measurement invariance analyses
library(tidyr)        # Required for reshaping data, especially converting data from wide to long format
library(tibble)       # Required for creating tidy tables/tibbles
library(vioplot)      # Required for split-violin plots of PPBS scores

## 1.2 Data-Import

In [ ]:
# WELCOME & Dutch data
# The original trial data are not publicly available.
# Place the required data files in a local, non-versioned data folder and adjust these paths as needed.
welcome_data_path <- file.path("data_raw", "WELCOME_DATA.csv")
dutch_data_path   <- file.path("data_raw", "ppbsgermanysent.sav")

data <- read.csv(
  welcome_data_path,
  fileEncoding = "UTF-8-BOM",
  stringsAsFactors = FALSE,
  check.names = FALSE
)

# Original Dutch data
original_data <- haven::read_sav(dutch_data_path)

## 1.3 Data-Preparation

In [ ]:
# Helper function: robust conversion to numeric
to_num <- function(x){
  if (is.numeric(x)) return(x)
  suppressWarnings(as.numeric(trimws(as.character(x))))
}

# Helper function for displaying readable HTML tables in Jupyter notebooks
display_table <- function(tab, caption, digits = 3, scroll = TRUE) {
  html_tab <- knitr::kable(
    tab,
    format  = "html",
    digits  = digits,
    caption = caption
  )
  if (scroll) {
    html_tab <- paste0(
      "<div style='overflow-x:auto; width:100%;'>",
      html_tab,
      "</div>"
    )
  }
  IRdisplay::display_html(html_tab)
}

# Convert ID to numeric
data$id_redcap <- to_num(data$id_redcap)

# Registration must be complete
data <- subset(data, registrierung_complete == 2)

# Group assignment (intervention / control)
data$gruppe <- ifelse(data$id_gruppe == 2, "Intervention",
                      ifelse(data$id_gruppe == 1, "Control", NA))
  
# -------------------
# Create PPBS dataset
# -------------------

ppbs_df <- data[, c(
  "id_redcap",
  "gruppe",
  
  # T0
  "t0_assessments_complete",
  "eltern_ppbs1_t0",
  "eltern_ppbs2_t0",
  "eltern_ppbs3_t0",
  "eltern_ppbs4_t0",
  "eltern_ppbs5_t0",
  "eltern_ppbs_score_t0",
  
  # T1
  "t1_assessments_teil_1_kind_bonding_complete",
  "eltern_ppbs1_t1",
  "eltern_ppbs2_t1",
  "eltern_ppbs3_t1",
  "eltern_ppbs4_t1",
  "eltern_ppbs5_t1",
  "eltern_ppbs_score_t1",
  
  # T2
  "t2_assessments_teil_1_kind_bonding_user_experience_complete",
  "eltern_ppbs1_t2",
  "eltern_ppbs2_t2",
  "eltern_ppbs3_t2",
  "eltern_ppbs4_t2",
  "eltern_ppbs5_t2",
  "eltern_ppbs_score_t2",
  
  # T3
  "t3_assessments_complete",
  "eltern_ppbs1_t3",
  "eltern_ppbs2_t3",
  "eltern_ppbs3_t3",
  "eltern_ppbs4_t3",
  "eltern_ppbs5_t3",
  "eltern_ppbs_score_t3"
), drop = FALSE]

# Convert all variables except group to numeric
num_cols <- setdiff(names(ppbs_df), "gruppe")
ppbs_df[num_cols] <- lapply(ppbs_df[num_cols], to_num)

# Recode 99 = "no response" to NA for PPBS items
ppbs_item_cols <- grep("^eltern_ppbs[1-5]_t[0-3]$", names(ppbs_df), value = TRUE)
ppbs_df[ppbs_item_cols] <- lapply(ppbs_df[ppbs_item_cols], function(x) {
  x[x == 99] <- NA
  x
})

# -------------------------------
# Useful lists for later analyses
# -------------------------------

ppbs_item_cols_by_time <- list(
  T0 = paste0("eltern_ppbs", 1:5, "_t0"),
  T1 = paste0("eltern_ppbs", 1:5, "_t1"),
  T2 = paste0("eltern_ppbs", 1:5, "_t2"),
  T3 = paste0("eltern_ppbs", 1:5, "_t3")
)

ppbs_complete_col_by_time <- list(
  T0 = "t0_assessments_complete",
  T1 = "t1_assessments_teil_1_kind_bonding_complete",
  T2 = "t2_assessments_teil_1_kind_bonding_user_experience_complete",
  T3 = "t3_assessments_complete"
)

ppbs_score_col_by_time <- list(
  T0 = "eltern_ppbs_score_t0",
  T1 = "eltern_ppbs_score_t1",
  T2 = "eltern_ppbs_score_t2",
  T3 = "eltern_ppbs_score_t3"
)
  
# -------------------
# DASS dataset for T0
# -------------------

dass_t0_df <- data[, c(
  "id_redcap",
  "t0_assessments_complete",
  "eltern_dass_depression_t0",
  "eltern_dass_angst_t0",
  "eltern_dass_stress_t0"
), drop = FALSE]

# Convert all variables to numeric
dass_t0_df[] <- lapply(dass_t0_df, to_num)
  
# -----------------------------------------------------------
# Combined T0 dataset for relations to psychological distress
# -----------------------------------------------------------

ppbs_dass_t0_df <- merge(
  ppbs_df[, c(
    "id_redcap",
    "t0_assessments_complete",
    "eltern_ppbs_score_t0"
  )],
  dass_t0_df[, c(
    "id_redcap",
    "eltern_dass_depression_t0",
    "eltern_dass_angst_t0",
    "eltern_dass_stress_t0"
  )],
  by = "id_redcap",
  all = FALSE
)

# Keep completed T0 assessments only
t0_complete <- to_num(ppbs_dass_t0_df$t0_assessments_complete) == 2
t0_complete[is.na(t0_complete)] <- FALSE

ppbs_dass_t0_df <- ppbs_dass_t0_df[t0_complete, , drop = FALSE]

# ------------------- 
# Colour scheme (LMU)
# -------------------

# RGB (0–255) -> hex, optionally with alpha
rgb255 <- function(r, g, b, a = 255) rgb(r, g, b, maxColorValue = 255, alpha = a)
col_lmu_gruen     <- rgb255(0,136,58)  
col_schwarz       <- rgb255(35,35,35) 
col_weiss         <- rgb255(255,255,255) 
col_akzent_orange <- rgb255(241,135,0)  
col_akzent_blau   <- rgb255(15,25,135)
col_akzent_cyan   <- rgb255(100,59,227)
col_akzent_violett<- rgb255(140,64,145)
col_akzent_rot    <- rgb255(215,25,25)
col_grau_dunkel   <- rgb255(98,100,104)
col_grau_mittel   <- rgb255(192,193,195)
col_grau_hell     <- rgb255(230,230,231)
col_grau_licht    <- rgb255(245,245,245)

# Palette 
lmu_pal <- c(
  "LMU Green"          = col_lmu_gruen,
  "Black"           = col_schwarz,
  "White"              = col_weiss,
  "Accent Orange"     = col_akzent_orange,
  "Accent Blue"       = col_akzent_blau,
  "Accent Cyan"       = col_akzent_cyan,
  "Accent Violet"    = col_akzent_violett,
  "Accent Red"        = col_akzent_rot,
  "Accent Dark Grey" = col_grau_dunkel,
  "Accent Medium Grey" = col_grau_mittel,
  "Accent Light Grey"   = col_grau_hell,
  "Accent Very Light Grey"  = col_grau_licht
)

# 2. Descriptive Statistics & Distribution Checks

## 2.1 Sample Characteristics

Aim: Generate console output for Table 1
- Family-level variables are calculated among families with completed T0 assessment
- Child-level variables are calculated among children from families with completed T0 assessment
- Categorical variables are displayed as n (%), with missing values shown as N/A
- NOTE: n differs between included families and included children because some families had twins

Output format:
- Continuous: mean ± SD; range
- Categorical: n (%)
- Print missing values as N/A

In [ ]:
# ----------------
# Helper functions
# ----------------

fmt_num <- function(x, digits = 2) {
  if (length(x) == 0 || is.na(x) || is.nan(x)) return("N/A")
  format(round(x, digits), nsmall = digits, trim = TRUE)
}

table1_groups <- c("Total", "Intervention", "Control")

get_table1_index <- function(gruppe, group_name) {
  gruppe <- as.character(gruppe)
  
  if (group_name == "Total") {
    rep(TRUE, length(gruppe))
  } else {
    gruppe == group_name
  }
}

print_continuous <- function(x, label, gruppe = NULL, digits = 2,
                             group_levels = table1_groups,
                             show_na = TRUE) {
  
  x <- to_num(x)
  
  if (is.null(gruppe)) {
    gruppe <- rep("Total", length(x))
    group_levels <- "Total"
  } else {
    gruppe <- as.character(gruppe)
  }
  
  cat("\n", label, "\n", sep = "")
  
  if (show_na) {
    cat("Group | N valid | mean ± SD; range | N/A\n")
  } else {
    cat("Group | N | mean ± SD; range\n")
  }
  
  for (g in group_levels) {
    
    idx <- get_table1_index(gruppe, g)
    
    x_group <- x[idx]
    denom <- sum(idx, na.rm = TRUE)
    
    n_missing <- sum(is.na(x_group))
    pct_missing <- if (denom > 0) n_missing / denom * 100 else NA_real_
    
    x_valid <- x_group[!is.na(x_group)]
    
    if (length(x_valid) == 0) {
      
      if (show_na) {
        cat(
          g, "| N valid = 0 | N/A | ",
          n_missing, " (", fmt_num(pct_missing, digits), "%)\n",
          sep = ""
        )
      } else {
        cat(g, "| N = 0 | N/A\n")
      }
      
    } else {
      
      if (show_na) {
        cat(
          g, "| N valid =", length(x_valid), "|",
          fmt_num(mean(x_valid), digits), "±", fmt_num(sd(x_valid), digits),
          ";", fmt_num(min(x_valid), digits), "-", fmt_num(max(x_valid), digits),
          "|", n_missing, " (", fmt_num(pct_missing, digits), "%)\n"
        )
      } else {
        cat(
          g, "| N =", length(x_valid), "|",
          fmt_num(mean(x_valid), digits), "±", fmt_num(sd(x_valid), digits),
          ";", fmt_num(min(x_valid), digits), "-", fmt_num(max(x_valid), digits), "\n"
        )
      }
    }
  }
}

print_categorical <- function(x, label, gruppe = NULL, value_labels = NULL,
                              digits = 2, group_levels = table1_groups,
                              show_na = TRUE) {
  
  x <- as.character(x)
  x[is.na(x) | x == ""] <- "N/A"
  
  if (is.null(gruppe)) {
    gruppe <- rep("Total", length(x))
    group_levels <- "Total"
  } else {
    gruppe <- as.character(gruppe)
  }
  
  # Define displayed categories
  if (!is.null(value_labels)) {
    categories <- names(value_labels)
    extra_categories <- setdiff(unique(x), categories)
    categories <- c(categories, extra_categories)
  } else {
    categories <- sort(unique(x))
  }
  
  # Always show N/A row if requested
  if (show_na && !"N/A" %in% categories) {
    categories <- c(categories, "N/A")
  }
  
  cat("\n", label, "\n", sep = "")
  cat("Category | ", paste(group_levels, collapse = " | "), "\n", sep = "")
  
  for (val in categories) {
    
    row_label <- if (!is.null(value_labels) && val %in% names(value_labels)) {
      value_labels[[val]]
    } else {
      val
    }
    
    cat(row_label)
    
    for (g in group_levels) {
      idx <- get_table1_index(gruppe, g)
      denom <- sum(idx, na.rm = TRUE)
      n_val <- sum(x[idx] == val, na.rm = TRUE)
      pct_val <- if (denom > 0) n_val / denom * 100 else NA_real_
      
      cat(" | ", n_val, " (", fmt_num(pct_val, digits), "%)", sep = "")
    }
    
    cat("\n")
  }
}

# --------------------
# Prepare source data
# --------------------

# Family-level working data
# Only families with completed T0 assessment are included
t0_complete <- to_num(data$t0_assessments_complete) == 2
t0_complete[is.na(t0_complete)] <- FALSE

table1_family <- data[t0_complete, , drop = FALSE]

# Child-level working data:
# Only variables that are available separately for child 1 / child 2 are included
# Children are included only if their family has a completed T0 assessment

table1_child <- rbind(
  data.frame(
    id_redcap = table1_family$id_redcap,
    gruppe = table1_family$gruppe,
    sex = to_num(table1_family$d_geschlecht_k1),
    birthweight = to_num(table1_family$d_gewichtgeburt_k1)
  ),
  data.frame(
    id_redcap = table1_family$id_redcap,
    gruppe = table1_family$gruppe,
    sex = to_num(table1_family$d_geschlecht_k2),
    birthweight = to_num(table1_family$d_gewichtgeburt_k2)
  ) 
)

# Remove empty second-child records
table1_child <- subset(table1_child, !is.na(sex) | !is.na(birthweight))

cat("TABLE 1: SAMPLE CHARACTERISTICS\n")
cat("Included families:", nrow(table1_family), "\n")
cat("Included children:", nrow(table1_child), "\n")
cat("\nIncluded families by group:\n")
print(table(table1_family$gruppe, useNA = "ifany"))
cat("\nIncluded children by group:\n")
print(table(table1_child$gruppe, useNA = "ifany"))

# ------------------------------------------------------------------------------
# Mother's age at birth in years

# REDCap: Text
# Recoding: Number
# ------------------------------------------------------------------------------

print_continuous(
  x      = table1_family$d_alter_mutter,
  gruppe = table1_family$gruppe,
  label  = "Mother's age at birth in years"
)

# ------------------------------------------------------------------------------
# Mother's citizenship

# REDCap: 1 Deutsch; 2 EU-Bürger oder EU-Bürgerin; 3 Drittstaat-Bürger oder Drittstaat-Bürgerin
# Recoding: German (=1); European Union (=2); Other (=3)
# ------------------------------------------------------------------------------

citizenship_rec <- to_num(table1_family$d_staat_mutter)
citizenship_rec[citizenship_rec == 99] <- NA

print_categorical(
  x = citizenship_rec,
  gruppe = table1_family$gruppe,
  label = "Mother's citizenship",
  value_labels = c(
    "1" = "German",
    "2" = "European Union",
    "3" = "Other"
  )
)

# ------------------------------------------------------------------------------
# Mother's language

# REDCap: 1 Deutsch; 2 Englisch; 3 Beides; 90 Keine von beiden
# Recoding: German (=1); English (=2); Both (= 3); Other (with translator) (=90)
# ------------------------------------------------------------------------------

language_raw <- to_num(table1_family$d_sprache_mutter)

print_categorical(
  x = language_raw,
  gruppe = table1_family$gruppe,
  label = "Mother's language",
  value_labels = c(
    "1" = "German",
    "2" = "English",
    "3" = "Both",
    "90" = "Other (with translator)"
  )
)

# ------------------------------------------------------------------------------
# Mother's education level

# Based on: d_schulabschluss_mutter

# REDCap:
# 1	Noch Schüler/Schülerin
# 2	Schule beendet ohne Abschluss
# 3	Volks- / Hauptschulabschluss bzw Polytechnische Oberschule mit Abschluss 8 oder 9 Klasse
# 4	Mittlere Reife / Realschulabschluss bzw Polytechnische Oberschule mit Abschluss 10 Klasse
# 5	Fachhochschulreife (Abschluss einer Fachoberschule etc.)
# 6	Abitur bzw. Erweiterte Oberschule mit Abschluss 12. Klasse (Hochschulreife)
# 90 Anderen Schulabschluss
# 99 Keine Angabe

# Recoding: 
# 1 + 2  -> No graduation
# 3      -> Graduation up to grade 9
# 4      -> Graduation up to grade 10
# 5 + 6  -> Graduation up to grade 12
# 90     -> Other
# 99     -> N/A
# ------------------------------------------------------------------------------

school_raw <- to_num(table1_family$d_schulabschluss_mutter)
school_raw[school_raw == 99] <- NA

# Recode education categories
school_rec <- rep(NA_character_, length(school_raw))

school_rec[school_raw %in% c(1, 2)] <- "no_graduation"
school_rec[school_raw == 3]         <- "grade_9"
school_rec[school_raw == 4]         <- "grade_10"
school_rec[school_raw %in% c(5, 6)] <- "grade_12"
school_rec[school_raw == 90]        <- "other"

print_categorical(
  x      = school_rec,
  gruppe = table1_family$gruppe,
  label  = "Mother's education level / Highest educational attainment",
  value_labels = c(
    "no_graduation" = "No graduation",
    "grade_9"       = "Graduation up to grade 9",
    "grade_10"      = "Graduation up to grade 10",
    "grade_12"      = "Graduation up to grade 12",
    "other"         = "Other",
    "N/A"           = "N/A"
  )
)

# ------------------------------------------------------------------------------
# Net monthly household income

# No recoding necessary
# ------------------------------------------------------------------------------

income_rec <- to_num(table1_family$d_einkommen_einfach)
income_rec[income_rec == 99] <- NA

print_categorical(
  x = income_rec,
  gruppe = table1_family$gruppe,
  label = "Net monthly household income",
  value_labels = c(
    "1" = "Below 2300€",
    "2" = "2300-5000€",
    "3" = "Over 5000€",
    "N/A" = "N/A"
  )
)

# ------------------------------------------------------------------------------
# Place of residence

# REDCap: 
# 1	Großstadt
# 2	Rand oder Vorort einer Großstadt
# 3	Mittel- oder Kleinstadt
# 4	Ländliches Dorf
# 5	Einzelgehöft oder alleinstehendes Haus auf dem Land
# 99 Keine Angabe

# Recoding:
# Urban (= 1 & 2)
# Town / semi-urban (=3)
# Rural (= 4 & 5)
# N/A (= 99)
# ------------------------------------------------------------------------------

wohnort <- to_num(table1_family$d_wohnort)

wohnort_rec <- ifelse(wohnort %in% c(1, 2), "Urban",
                      ifelse(wohnort == 3, "Town / semi-urban",
                             ifelse(wohnort %in% c(4, 5), "Rural",
                                    ifelse(wohnort == 99, "N/A", NA))))

print_categorical(
  x = wohnort_rec,
  gruppe = table1_family$gruppe,
  label = "Place of residence",
  value_labels = c("N/A" = "N/A")
)

# ------------------------------------------------------------------------------
# Previous children

# REDCap: 1 Ja; 2 Nein
# Recoding: Yes -> multiparous (= 1); No -> first-time mother (= 2)
# ------------------------------------------------------------------------------

prev_children <- to_num(table1_family$d_geschwister)

print_categorical(
  x = prev_children,
  gruppe = table1_family$gruppe,
  label = "Previous children",
  value_labels = c(
    "1" = "Yes -> multiparous",
    "2" = "No -> first-time mother"
  )
)

# ------------------------------------------------------------------------------
# Twins

# REDCap ("Geht es um Mehrlinge?"): 1 Nein; 2 Ja, 2; 3 Ja, 3; 4 Ja, 4
# Recoding ("Twins?"): Yes (= 2,3,4); No (= 1)
# NOTE: This variable is labelled as "Twins" because no triplets or higher-order multiples were present in the current sample.
# ------------------------------------------------------------------------------

twins_raw <- to_num(table1_family$d_mehrlinge)

twins_rec <- ifelse(twins_raw == 1, "No",
                    ifelse(twins_raw %in% c(2, 3, 4), "Yes", NA))

print_categorical(
  x = twins_rec,
  gruppe = table1_family$gruppe,
  label = "Twins"
)

# ------------------------------------------------------------------------------
# Sex (child-level)

# No recoding necessary
# ------------------------------------------------------------------------------

print_categorical(
  x = table1_child$sex,
  gruppe = table1_child$gruppe,
  label = "Sex",
  value_labels = c(
    "1" = "Female",
    "2" = "Male",
    "3" = "Diverse"
  )
)

# ------------------------------------------------------------------------------
# Gestational age category

# No recoding necessary
# ------------------------------------------------------------------------------

gest_age <- to_num(table1_family$id_gestationsalter)

print_categorical(
  x = gest_age,
  gruppe = table1_family$gruppe,
  label = "Gestational age category",
  value_labels = c(
    "1" = "Born with 20-29 weeks (Z3A.2)",
    "2" = "Born with 30-39 weeks (Z3A.3)",
    "3" = "Born with >40 weeks (Z3A.4)"
  )
)

# ------------------------------------------------------------------------------
# Birth weight category (child-level)

# REDCap: Text
# Recoding: <500 g; 500-999 g; 1000-2499 g; ≥2500 g
# ------------------------------------------------------------------------------

bw <- to_num(table1_child$birthweight)

bw_rec <- ifelse(is.na(bw), NA,
                 ifelse(bw < 500, "<500 g",
                        ifelse(bw < 1000, "500-999 g",
                               ifelse(bw < 2500, "1000-2499 g", "≥2500 g"))))

print_categorical(
  x = bw_rec,
  gruppe = table1_child$gruppe,
  label = "Birth weight category"
)

# ------------------------------------------------------------------------------
# Top 5 side diagnoses and complications by frequency

# No recoding necessary
# Format: n (%)
# Denominator: child level, because the variable is available for k1 and k2
# ------------------------------------------------------------------------------

# Labels for checkbox categories
diag_labels <- c(
  "1"  = "[D18] Hemangioma / other benign vascular tumours of the skin",
  "2"  = "[D22] Naevus",
  "3"  = "[E83.5] Nephrocalcinosis / disorders of calcium metabolism",
  "4"  = "[H10;N39.0;P23;P39] Other infections / inflammations",
  "5"  = "[H35.1;H35.6] Retinopathy / retinal haemorrhage",
  "6"  = "[H90;H91] Hearing disorder",
  "7"  = "[I27;P29.3] Pulmonary hypertension / right heart strain",
  "8"  = "[I82] Thrombosis",
  "9"  = "[K21;P78] Gastro-oesophageal reflux / other digestive disorders",
  "10" = "[K40-K46] Hernias",
  "11" = "[K91.2] Short bowel syndrome",
  "12" = "[L00-L99] Other dermatological problems / skin disorders",
  "13" = "[L20-L30] Exanthema / erythema / rash / diaper dermatitis",
  "14" = "[N43;Q55.2] Hydrocele, undescended testis / other urogenital malformations",
  "15" = "[P05] SGA / intrauterine growth restriction",
  "16" = "[P10;P11;P52;P91] Cerebral haemorrhage / stroke / PVL",
  "17" = "[P12] Cephalohematoma / other birth-related head soft tissue injuries",
  "18" = "[P22;P27.1] Respiratory distress / BPD / breathing problems",
  "19" = "[P24.0] Meconium aspiration syndrome",
  "20" = "[P25.1] Pneumothorax",
  "21" = "[P28.3;P28.4] Apnoea-bradycardia symptoms / desaturation episodes",
  "22" = "[P28.8;Q31.5] Stridor / laryngomalacia",
  "23" = "[P29.0;P29.2] Cardiovascular diseases",
  "24" = "[P36] Sepsis",
  "25" = "[P38] Omphalitis / umbilical infection",
  "26" = "[P59] Hyperbilirubinaemia / neonatal jaundice",
  "27" = "[P61;P55] Anaemia / thrombocytopenia / haemolytic disease",
  "28" = "[P70] Hypo- or hyperglycaemia",
  "29" = "[P74] Electrolyte disorder",
  "30" = "[P77] Necrotising enterocolitis",
  "31" = "[P81] Hypothermia / temperature instability",
  "32" = "[P90] Cerebral seizures",
  "33" = "[P92;P92.2] Feeding disorder / feeding intolerance / poor sucking",
  "34" = "[P94] Other neurological disorder",
  "35" = "[P96.1] Neonatal withdrawal syndrome",
  "36" = "[Q00-Q99] Malformations / genetic disorders in general",
  "37" = "[Q20-Q28] Structural heart defects / abnormal fetal cardiac findings",
  "38" = "[Q21.1;Q25.0] PDA / PFO / atrial septal defect",
  "39" = "[Q33.6] Pulmonary hypoplasia",
  "40" = "[Q60-Q64] Abnormal renal findings / other renal malformations",
  "41" = "[Q65;Q68] Orthopaedic malformations / congenital musculoskeletal deformities",
  "42" = "[Q91;Q03.3;Z98.2] Hydrocephalus / VP shunt",
  "43" = "[R62] Failure to thrive",
  "44" = "[T78] Allergies / allergic reactions",
  "45" = "[Z22.3] MRSA / carrier of pathogenic bacteria",
  "46" = "[Z93.2-3] Presence of ileostomy / colostomy",
  "90" = "Other"
)

# Select checkbox columns for child 1 and child 2
diag_k1_cols <- grep("^d_neben2_komp2_k1___", names(table1_family), value = TRUE)
diag_k2_cols <- grep("^d_neben2_komp2_k2___", names(table1_family), value = TRUE)

# Helper function: transform one child's checkbox variables into long format
extract_diag_long <- function(df, cols, child_label) {
  tmp <- df[, cols, drop = FALSE]
  
  # Convert all variables to numeric
  tmp[] <- lapply(tmp, to_num)
  
  # Convert to long format
  long <- stack(tmp)
  names(long) <- c("checked", "variable")
  
  # Add row ID to recover group assignment
  long$row_id <- rep(seq_len(nrow(tmp)), times = length(cols))
  long$gruppe <- df$gruppe[long$row_id]
  
  # Keep checked categories only
  long <- subset(long, checked == 1)
  
  # Extract category from column name
  long$code <- sub(".*___", "", long$variable)
  long$child <- child_label
  
  # Exclude "none" (0)
  long <- subset(long, code != "0")
  
  long
}

diag_long_k1 <- extract_diag_long(table1_family, diag_k1_cols, "k1")
diag_long_k2 <- extract_diag_long(table1_family, diag_k2_cols, "k2")

diag_long_all <- rbind(diag_long_k1, diag_long_k2)

# Denominator by child and group
child_diag_den <- rbind(
  data.frame(
    gruppe = table1_family$gruppe,
    child = "k1",
    child_exists = !is.na(to_num(table1_family$d_geschlecht_k1)) | 
      !is.na(to_num(table1_family$d_gewichtgeburt_k1)),
    has_diag_block = rowSums(!is.na(table1_family[, diag_k1_cols, drop = FALSE])) > 0
  ),
  data.frame(
    gruppe = table1_family$gruppe,
    child = "k2",
    child_exists = !is.na(to_num(table1_family$d_geschlecht_k2)) | 
      !is.na(to_num(table1_family$d_gewichtgeburt_k2)),
    has_diag_block = rowSums(!is.na(table1_family[, diag_k2_cols, drop = FALSE])) > 0
  )
)

child_diag_den <- subset(child_diag_den, child_exists & has_diag_block)

# Calculate overall frequencies
diag_freq <- as.data.frame(table(diag_long_all$code), stringsAsFactors = FALSE)
names(diag_freq) <- c("code", "n")

diag_freq$label <- diag_labels[diag_freq$code]

# Sort by frequency and select the overall top 5
diag_top5 <- diag_freq[order(-diag_freq$n, diag_freq$code), ]
diag_top5 <- head(diag_top5, 5)

# Console output
cat("\nTop 5 side diagnoses and complications\n")
cat("Diagnosis | Total | Intervention | Control\n")

for (i in seq_len(nrow(diag_top5))) {
  
  code_i  <- diag_top5$code[i]
  label_i <- diag_top5$label[i]
  
  cat(label_i)
  
  for (g in table1_groups) {
    
    if (g == "Total") {
      n_den <- nrow(child_diag_den)
      n_num <- sum(diag_long_all$code == code_i, na.rm = TRUE)
    } else {
      n_den <- sum(child_diag_den$gruppe == g, na.rm = TRUE)
      n_num <- sum(diag_long_all$code == code_i & diag_long_all$gruppe == g, na.rm = TRUE)
    }
    
    pct <- if (n_den > 0) n_num / n_den * 100 else NA_real_
    
    cat(" | ", n_num, " (", fmt_num(pct, 2), "%)", sep = "")
  }
  
  cat("\n")
}

## 2.2 OPTIONAL: PPBS Split-Violin Plots (T0–T3)

Aim: To visually inspect the distribution of PPBS sum scores and the corresponding group sizes across all assessment time points.

The function:
- Uses the time points defined in ppbs_score_col_by_time
- Filters each panel to complete == 2
- Splits by group
- Converts relevant values with to_num()
- Draws split violins including individual points and n labels
  
NOTE: Unlike the descriptive summary tables below, this plot includes both study groups at all time points.

In [ ]:
plot_ppbs_split_violins <- function(data,
                                    group_col    = "gruppe",
                                    group_left   = "Intervention",
                                    group_right  = "Control",
                                    y_max        = 15,
                                    point_jitter = 0.02,
                                    point_cex    = 0.9,
                                    point_alpha  = 0.4) {
  
  # Time points and corresponding columns
  panels <- names(ppbs_score_col_by_time)
  
  # Define colours
  col_left_base    <- lmu_pal["Accent Orange"]
  col_right_base   <- "grey40"
  
  col_left_fill    <- adjustcolor(col_left_base, 0.40)
  col_left_border  <- adjustcolor(col_left_base, 0.80)
  col_left_point   <- adjustcolor(col_left_base, point_alpha)
  
  col_right_fill   <- adjustcolor("grey60", 0.40)
  col_right_border <- "grey40"
  col_right_point  <- adjustcolor(col_right_base, point_alpha)
  
  # Prepare layout
  op <- par(no.readonly = TRUE)
  on.exit(par(op))
  par(mfrow = c(1, length(panels)),
      mar  = c(6, 3, 4, 1) + 0.1,
      oma  = c(0, 3, 3.5, 0))
  
  draw_panel <- function(df, comp_col, val_col, panel_label) {
    
    # Complete cases only
    ok <- to_num(df[[comp_col]]) == 2
    ok[is.na(ok)] <- FALSE
    df <- df[ok, , drop = FALSE]
    
    # Split groups
    L <- df[df[[group_col]] == group_left,  , drop = FALSE]
    R <- df[df[[group_col]] == group_right, , drop = FALSE]
    
    vL <- to_num(L[[val_col]]); vL <- vL[is.finite(vL)]
    vR <- to_num(R[[val_col]]); vR <- vR[is.finite(vR)]
    
    nL <- length(vL)
    nR <- length(vR)
    
    # Empty panel
    plot(NA, xlim = c(0.5, 1.5), ylim = c(0, y_max),
         xaxt = "n", xlab = "", ylab = "", yaxt = "none")
    axis(2)
    title(panel_label)
    axis(1, at = 1, labels = paste0(group_left, " vs. ", group_right), line = 0)
    
    # n label below the x-axis
    usr <- par("usr")
    y_n <- grconvertY(
      grconvertY(usr[3], from = "user", to = "ndc") - 0.07,
      from = "ndc", to = "user"
    )
    opx <- par(xpd = NA)
    text(c(0.90, 1.10), y_n,
         labels = c(sprintf("[n=%d]", nL), sprintf("[n=%d]", nR)),
         cex = 0.85, col = "grey30")
    par(opx)
    
    # Left half (Intervention)
    vioplot(
      vL, side = "left", add = TRUE, at = 1,
      col    = col_left_fill,
      border = col_left_border,
      plotCentre = "line"
    )
    set.seed(123)
    points(
      jitter(rep(0.90, nL), amount = point_jitter), vL,
      pch = 19,
      col = col_left_point,
      cex = point_cex
    )
    
    # Right half (Control)
    vioplot(
      vR, side = "right", add = TRUE, at = 1,
      col    = col_right_fill,
      border = col_right_border,
      plotCentre = "line"
    )
    set.seed(124)
    points(
      jitter(rep(1.10, nR), amount = point_jitter), vR,
      pch = 19,
      col = col_right_point,
      cex = point_cex
    )
  }
  
  # Draw all time points
  for (p in panels) {
    draw_panel(
      data,
      comp_col    = ppbs_complete_col_by_time[[p]],
      val_col     = ppbs_score_col_by_time[[p]],
      panel_label = p
    )
  }
  
  mtext("PPBS Sum Score", side = 2, outer = TRUE, line = 1.2)
  mtext("PPBS – T0 to T3",
        side = 3, outer = TRUE, line = 1.1, cex = 1.2, font = 2)
}

# Function call
plot_ppbs_split_violins(ppbs_df)

## 2.3 PPBS Item & Sum Score Distributions

In [ ]:
# -----------------
# --- 1. Helper ---
# -----------------

# Helper function to calculate skewness using the third standardized moment
#   - 0: symmetric
#   - Positive value: right-skewed distribution
#   - Negative value: left-skewed distribution; many parents have high bonding scores
skewness <- function(x, na.rm = TRUE) {
  x <- as.numeric(x)
  if (na.rm) x <- x[!is.na(x)]
  n <- length(x)
  if (n < 3) return(NA_real_) 
  m <- mean(x)
  s <- sd(x)
  if (is.na(s) || s == 0) return(0) 
  mean(((x - m) / s)^3)
}

# Helper function to calculate excess kurtosis using the fourth standardized moment minus 3
#   - 0: normal distribution (mesokurtic)
#   - Positive value: leptokurtic; values are strongly concentrated around the centre
#   - Negative value: platykurtic; the distribution is relatively broad and flat
kurtosis <- function(x, na.rm = TRUE) {
  x <- as.numeric(x)
  if (na.rm) x <- x[!is.na(x)]
  n <- length(x)
  if (n < 4) return(NA_real_)
  m <- mean(x)
  s <- sd(x)
  if (is.na(s) || s == 0) return(0)
  mean(((x - m) / s)^4) - 3
}

# Helper function to calculate floor and ceiling effects using an 80% criterion
#   - Floor effect: many participants select the lowest possible value
#   - Ceiling effect: many participants select the highest possible value
#   - 80% criterion: a flag is assigned when at least 80% select the same extreme value
#   - Ceiling flag = TRUE: the item no longer differentiates well at the upper end
floor_ceiling_props <- function(x, xmin, xmax, threshold = 0.80) {
  x <- x[!is.na(x)]
  p_floor <- mean(x == xmin)
  p_ceil  <- mean(x == xmax)
  list(
    floor = p_floor,
    ceiling = p_ceil,
    floor_flag = p_floor >= threshold,
    ceiling_flag = p_ceil >= threshold
  )
}

# --------------------------------------------------------------
# --- 2. Table: Sum-score distribution by time point (T0–T3) ---
# --------------------------------------------------------------

ppbs_summary_by_time <- function(data) {
  panels <- names(ppbs_score_col_by_time)
  
  res_list <- lapply(panels, function(p) {
    comp_col <- ppbs_complete_col_by_time[[p]]
    val_col  <- ppbs_score_col_by_time[[p]]
    
    # Filter to completed assessments
    # T0: all complete cases
    # T1-T3: control group only
    if (p == "T0") {
      ok <- to_num(data[[comp_col]]) == 2
    } else {
      ok <- to_num(data[[comp_col]]) == 2 & data$gruppe == "Control"
    }
    raw_vals <- to_num(data[[val_col]][ok])
    
    # Count missing values
    n_total   <- length(raw_vals)
    n_missing <- sum(is.na(raw_vals))
    x         <- raw_vals[!is.na(raw_vals)]
    n_valid   <- length(x)
    
    # Calculate statistics
    p_sw <- if (n_valid >= 3 && n_valid <= 5000) shapiro.test(x)$p.value else NA_real_
    # IMPORTANT: xmin = 0 and xmax = 15 for the sum score
    fc   <- floor_ceiling_props(x, xmin = 0, xmax = 15, threshold = 0.80)
    
    tibble::tibble(
      scale        = "PPBS Sum Score",
      timepoint    = p,
      n_total      = n_total,   # Denominator: completed assessment
      n_valid      = n_valid,   # Actual n used for calculation
      n_missing    = n_missing, # Number of missing PPBS sum scores among completed assessments
      prop_missing = n_missing / n_total,
      # Measures of location and dispersion
      min          = min(x),
      q25          = quantile(x, 0.25, names = FALSE),
      median       = median(x),
      mean         = mean(x),
      q75          = quantile(x, 0.75, names = FALSE),
      iqr          = IQR(x),
      max          = max(x),
      sd           = sd(x),
      # Distributional shape measures and flags
      skewness     = skewness(x, na.rm = FALSE),
      kurtosis     = kurtosis(x, na.rm = FALSE),
      prop_floor   = fc$floor,
      prop_ceiling = fc$ceiling,
      floor_80     = fc$floor_flag,
      ceiling_80   = fc$ceiling_flag,
      p_shapiro    = p_sw
    )
  })
  
  dplyr::bind_rows(res_list)
}

# Function call
tab_ppbs_by_time <- ppbs_summary_by_time(ppbs_df)

# For display purposes in this Jupyter notebook, the full sum-score table is split into two smaller tables:
# (1) main descriptive statistics and (2) distributional characteristics.
# The underlying object tab_ppbs_by_time still contains the complete output.

cat("\nNOTE: For readability in this Jupyter notebook, the complete PPBS sum-score table is displayed as two smaller tables.\n")
cat("The underlying object 'tab_ppbs_by_time' contains the full output.\n\n")

tab_ppbs_by_time_main <- tab_ppbs_by_time %>%
  dplyr::select(
    timepoint, n_total, n_valid, n_missing,
    min, q25, median, mean, q75, iqr, max, sd
  )
display_table(
  tab_ppbs_by_time_main,
  caption = "PPBS sum-score descriptives across assessment time points",
  digits = 3
)

tab_ppbs_by_time_distribution <- tab_ppbs_by_time %>%
  dplyr::select(
    timepoint, skewness, kurtosis,
    prop_floor, prop_ceiling,
    floor_80, ceiling_80, p_shapiro
  )
display_table(
  tab_ppbs_by_time_distribution,
  caption = "Distributional characteristics of PPBS sum scores across assessment time points",
  digits = 3
)

# -------------------------------------------------
# --- 3. Table: Item distribution by time point ---
# -------------------------------------------------

ppbs_item_summary_by_time <- function(df, threshold = 0.80) {
  times <- names(ppbs_item_cols_by_time)
  
  res <- lapply(times, function(tp) {
    comp_col <- ppbs_complete_col_by_time[[tp]]
    items    <- ppbs_item_cols_by_time[[tp]]
    
    # Only participants who completed the respective assessment time point
    # T0: all complete cases
    # T1-T3: control group only
    if (tp == "T0") {
      dft <- df[to_num(df[[comp_col]]) == 2, items, drop = FALSE]
    } else {
      dft <- df[to_num(df[[comp_col]]) == 2 & df$gruppe == "Control", items, drop = FALSE]
    }
    
    out <- lapply(items, function(v) {
      raw_x <- to_num(dft[[v]])
      
      n_total   <- length(raw_x)
      n_missing <- sum(is.na(raw_x))
      x         <- raw_x[!is.na(raw_x)]
      n_valid   <- length(x)
      
      p_sw <- if (n_valid >= 3 && n_valid <= 5000) shapiro.test(x)$p.value else NA_real_
      # IMPORTANT: xmin = 0 and xmax = 3 for the items
      fc   <- floor_ceiling_props(x, xmin = 0, xmax = 3, threshold = threshold)
      
      tibble::tibble(
        timepoint    = tp,
        item         = v,
        n_total      = n_total,
        n_valid      = n_valid,
        n_missing    = n_missing,
        prop_missing = n_missing / n_total,
        # Measures of location and dispersion
        min          = min(x),
        q25          = quantile(x, 0.25, names = FALSE),
        median       = median(x),
        mean         = mean(x),
        q75          = quantile(x, 0.75, names = FALSE),
        iqr          = IQR(x),
        max          = max(x),
        sd           = sd(x),
        # Distributional shape measures and validity flags:
        skewness     = skewness(x, na.rm = FALSE),
        kurtosis     = kurtosis(x, na.rm = FALSE),
        prop_floor   = fc$floor,
        prop_ceiling = fc$ceiling,
        floor_80     = fc$floor_flag,
        ceiling_80   = fc$ceiling_flag,
        p_shapiro    = p_sw
      )
    })
    
    dplyr::bind_rows(out)
  })
  
  dplyr::bind_rows(res)
}

# Function call
tab_ppbs_items_by_time <- ppbs_item_summary_by_time(ppbs_df)

# For display purposes in this Jupyter notebook, the full item-level table is split into two smaller tables:
# (1) main item descriptives and (2) distributional characteristics.
# The underlying object tab_ppbs_items_by_time still contains the complete output.

cat("\nNOTE: For readability in this Jupyter notebook, the complete PPBS item-level table is displayed as two smaller tables.\n")
cat("The underlying object 'tab_ppbs_items_by_time' contains the full output.\n\n")

tab_ppbs_items_main <- tab_ppbs_items_by_time %>%
  dplyr::select(
    timepoint, item, n_total, n_valid, n_missing,
    min, q25, median, mean, q75, iqr, max, sd
  )
display_table(
  tab_ppbs_items_main,
  caption = "PPBS item-level descriptives across assessment time points",
  digits = 3
)

tab_ppbs_items_distribution <- tab_ppbs_items_by_time %>%
  dplyr::select(
    timepoint, item, skewness, kurtosis,
    prop_floor, prop_ceiling,
    floor_80, ceiling_80, p_shapiro
  )
display_table(
  tab_ppbs_items_distribution,
  caption = "Distributional characteristics of PPBS items across assessment time points",
  digits = 3
)

# -----------------------------------
# --- 4. Histograms by time point ---
# -----------------------------------

plot_ppbs_hist_by_time <- function(data,
                                   main_title = "PPBS – distribution by timepoint",
                                   breaks     = seq(-0.5, 15.5, by = 1),
                                   xlim       = c(0, 15),
                                   ylim       = c(0, 50)) {
  
  panels <- names(ppbs_score_col_by_time)
  
  # Prepare data for all time points
  # T0: all complete cases
  # T1-T3: control group only
  x_list <- lapply(panels, function(p) {
    comp_col <- ppbs_complete_col_by_time[[p]]
    val_col  <- ppbs_score_col_by_time[[p]]
    
    if (p == "T0") {
      ok <- to_num(data[[comp_col]]) == 2
    } else {
      ok <- to_num(data[[comp_col]]) == 2 & data$gruppe == "Control"
    }
    
    x <- to_num(data[[val_col]][ok])
    x[is.finite(x)]
  })
  names(x_list) <- panels
  
  # Set layout
  op <- par(mfrow = c(1, length(panels)),
            mar  = c(4.5, 4.5, 4, 1) + 0.1,
            oma  = c(3, 4, 3, 0)) 
  on.exit(par(op))            
  
  # Draw one histogram for each time point
  for (p in panels) {
    x <- x_list[[p]]
    n <- length(x)
    
    # Compute histogram object based on counts
    h <- hist(x, breaks = breaks, plot = FALSE)
    
    # counts -> percent
    pct <- if (sum(h$counts) > 0) 100 * h$counts / sum(h$counts) else h$counts
    
    # Histogram plotten
    h$counts <- pct
    plot(h,
         freq = TRUE,
         main = paste0(p, "  [n = ", n, "]"),
         xlab = "",
         ylab = "",
         xlim = xlim,
         ylim = ylim,
         col  = adjustcolor("grey40", 0.6),
         border = "white")
  }
  
  # Shared overall title
  mtext(main_title, side = 3, outer = TRUE, line = 1, cex = 1.1, font = 2)
  # Shared y-axis label
  mtext("Relative frequency [%]", side = 2, outer = TRUE, line = 1.8, cex = 0.9)
  # Shared x-axis label
  mtext("PPBS Sum Score", side = 1, outer = TRUE, line = 1.5, cex = 0.9)
}

# Function call
plot_ppbs_hist_by_time(ppbs_df)

# ---------------------------------
# --- 5. Boxplots by time point ---
# ---------------------------------

plot_ppbs_boxplots_by_time <- function(data,
                                       main_title = "PPBS – boxplots by timepoint",
                                       ylim = c(0, 15)) {
  
  panels <- names(ppbs_score_col_by_time)
  
  # Prepare data
  # T0: all complete cases
  # T1-T3: control group only
  x_list <- lapply(panels, function(p) {
    comp_col <- ppbs_complete_col_by_time[[p]]
    val_col  <- ppbs_score_col_by_time[[p]]
    
    if (p == "T0") {
      ok <- to_num(data[[comp_col]]) == 2
    } else {
      ok <- to_num(data[[comp_col]]) == 2 & data$gruppe == "Control"
    }
    
    x <- to_num(data[[val_col]][ok])
    x[is.finite(x)]
  })
  names(x_list) <- panels
  
  # Layout
  op <- par(mfrow = c(1, length(panels)),
            mar  = c(4.5, 4.5, 4, 1) + 0.1,
            oma  = c(3, 4, 3, 0))
  on.exit(par(op))
  
  # Draw boxplots
  for (p in panels) {
    x <- x_list[[p]]
    n <- length(x)
    
    boxplot(x,
            main = paste0(p, "  [n = ", n, "]"),
            ylim = ylim,
            ylab = "",
            xlab = "",
            col  = "grey85",
            border = "grey40",
            outline = TRUE)
  }
  
  # Shared title and y-axis
  mtext(main_title, side = 3, outer = TRUE, line = 1, cex = 1.1, font = 2)
  mtext("PPBS Sum Score", side = 2, outer = TRUE, line = 1.8, cex = 0.9)
}

# Function call
plot_ppbs_boxplots_by_time(ppbs_df)

# ------------------------------------------------
# --- 6. Raincloud-style plot: PPBS sum scores ---
# ------------------------------------------------

ppbs_sum_long <- purrr::map_dfr(names(ppbs_score_col_by_time), function(tp) {
  
  comp_col  <- ppbs_complete_col_by_time[[tp]]
  score_col <- ppbs_score_col_by_time[[tp]]
  
  tmp <- tibble::tibble(
    timepoint = tp,
    gruppe    = ppbs_df$gruppe,
    score     = to_num(ppbs_df[[score_col]]),
    complete  = to_num(ppbs_df[[comp_col]])
  ) %>%
    dplyr::filter(complete == 2, !is.na(score))
  
  if (tp != "T0") {
    tmp <- tmp %>% dplyr::filter(gruppe == "Control")
  }
  
  tmp
})

# Set order of time points
ppbs_sum_long$timepoint <- factor(
  ppbs_sum_long$timepoint,
  levels = c("T0", "T1", "T2", "T3")
)

# n per time point for axis labels
n_labels <- ppbs_sum_long %>%
  dplyr::count(timepoint) %>%
  dplyr::mutate(label = paste0(timepoint, "\n(n = ", n, ")"))

axis_labels <- setNames(n_labels$label, n_labels$timepoint)

# Raincloud-style plot: note that a violin may be questionable for discrete data --> Plot without violin
ggplot(ppbs_sum_long, aes(x = timepoint, y = score)) +
  geom_boxplot(
    width = 0.18,
    outlier.shape = NA,
    alpha = 0.8
  ) +
  geom_jitter(
    width = 0.08,
    height = 0,
    alpha = 0.25,
    size = 1.4
  ) +
  scale_x_discrete(labels = axis_labels) +
  scale_y_continuous(
    limits = c(0, 15),
    breaks = 0:15
  ) +
  labs(
    title = "PPBS sum score distributions across assessment time points",
    subtitle = "T0: full sample; T1-T3: control group only",
    x = "Timepoint",
    y = "PPBS sum score"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold"),
    panel.grid.minor = element_blank()
  )

# 3. Factor Suitability & Correlations

Aims of this section:
- Prepare PPBS items at T0 for structural validity analyses
- Descriptive and graphical exploration of inter-item associations
- Conduct preliminary checks relevant for factor analysis
- Calculate Spearman and polychoric correlation matrices

In [ ]:
# -----------------------------
# --- 3.1 Create T0 dataset ---
# -----------------------------

# Select cases with completed T0 assessment
t0_complete <- to_num(ppbs_df$t0_assessments_complete) == 2
t0_complete[is.na(t0_complete)] <- FALSE

items_t0_raw <- ppbs_df[
  t0_complete,
  ppbs_item_cols_by_time$T0,
  drop = FALSE
]

# Number of cases with completed T0 assessment
n_t0_assessment_complete <- nrow(items_t0_raw)

# Listwise deletion for factor-analytic checks: rows with at least one missing PPBS item are removed
items_t0 <- na.omit(items_t0_raw)

# Assign shorter item names
colnames(items_t0) <- paste0("PPBS", 1:5)

# Number of complete item-level cases
n_t0_item_complete <- nrow(items_t0)

cat("\nT0 item dataset for factor-analytic checks\n")
cat("N with completed T0 assessment:", n_t0_assessment_complete, "\n")
cat("N with complete PPBS item data:", n_t0_item_complete, "\n")
cat("N excluded because of at least one missing PPBS item:",
    n_t0_assessment_complete - n_t0_item_complete, "\n")

# ---------------------------------------------------
# --- 3.2 Visual exploration of item associations ---
# ---------------------------------------------------

psych::pairs.panels(
  items_t0,
  jiggle = TRUE,
  method = "spearman",
  lm = TRUE,
  main = "PPBS T0: Visual item associations"
)

# ---------------------------------------------
# --- 3.3 Spearman correlation matrix at T0 ---
# ---------------------------------------------

R_spearman_t0 <- cor(
  items_t0,
  use = "complete.obs",
  method = "spearman"
)

cat("\nPPBS T0: Spearman item correlation matrix\n")
print(round(R_spearman_t0, 2))

psych::corPlot(
  R_spearman_t0,
  main = "PPBS T0: Spearman item correlations"
)

# -----------------------------------------------
# --- 3.4 Polychoric correlation matrix at T0 ---
# -----------------------------------------------

R_poly_t0 <- psych::polychoric(items_t0)$rho

cat("\nPPBS T0: Polychoric item correlation matrix\n")
print(round(R_poly_t0, 2))

psych::corPlot(
  R_poly_t0,
  main = "PPBS T0: Polychoric item correlations"
)

In [ ]:
# ------------------------------------------------------------
# --- 3.5 Preliminary check of the item correlation matrix ---
# ------------------------------------------------------------

# Determinant of the Spearman correlation matrix
# Aim: Should not be too small; used to rule out extreme multicollinearity

det_spearman_t0 <- det(R_spearman_t0)

cat("\nPPBS T0: Determinant of the Spearman item correlation matrix\n")
cat("Determinant =", format(det_spearman_t0, scientific = FALSE), "\n")

In [ ]:
# ------------------------------------------------------------------
# --- OPTIONAL: Item correlation checks for follow-up timepoints ---
# ------------------------------------------------------------------

item_checks_followup_timepoints <- function(df) {
  
  times <- setdiff(names(ppbs_item_cols_by_time), "T0")
  
  for (tp in times) {
    
    comp_col  <- ppbs_complete_col_by_time[[tp]]
    item_cols <- ppbs_item_cols_by_time[[tp]]
    
    # T1-T3: control group only and complete cases
    dft <- df[
      to_num(df[[comp_col]]) == 2 & df$gruppe == "Control",
      item_cols,
      drop = FALSE
    ]
    
    # Listwise deletion
    dft <- na.omit(dft)
    colnames(dft) <- paste0("PPBS", 1:5)
    
    n_tp <- nrow(dft)
    
    cat("\n------------------------------------\n")
    cat("ITEM CHECKS:", tp, "\n")
    cat("N complete cases:", n_tp, "\n")
    
    if (n_tp < 3) {
      cat("Not enough complete cases for correlation analyses.\n")
      next
    }
    
    # Spearman correlations
    R_spear <- cor(dft, use = "complete.obs", method = "spearman")
    
    # Polychoric correlations
    R_poly <- psych::polychoric(dft)$rho
    
    # Determinant of the Spearman correlation matrix
    det_res <- det(R_spear)
    
    cat("\nSpearman correlations:\n")
    print(round(R_spear, 2))
    
    cat("\nPolychoric correlations:\n")
    print(round(R_poly, 2))
    
    cat("\nDeterminant of the Spearman item correlation matrix:\n")
    cat("Determinant =", format(det_res, scientific = FALSE), "\n")
  }
}

# Function call
item_checks_followup_timepoints(ppbs_df)

# 4. Confirmatory Factor Analysis (CFA)

CFA approach: 
- Aim: Confirmation of the unidimensional structure described in the literature (1 Factor „Bonding“)
- CFA is performed for T0 only because this is the largest available sample
- Model: One latent factor with five indicators (PPBS1–PPBS5)
- Data are ordinal (0–3) → CFA with an ordinal estimator (WLSMV)
- Declare items as ordered variables (lavaan: ordered=…)
- Use the items_t0 subset created in Section 3

In [ ]:
# -------------------------------
# --- 4.1 Model specification ---
# -------------------------------

ppbs_model <- 'Bonding =~ PPBS1 + PPBS2 + PPBS3 + PPBS4 + PPBS5'

# ------------------------------
# --- 4.2 Estimate CFA model ---
# ------------------------------

fit_cfa_t0 <- lavaan::cfa(
  model     = ppbs_model,
  data      = items_t0,
  ordered   = c("PPBS1", "PPBS2", "PPBS3", "PPBS4", "PPBS5"),
  estimator = "WLSMV"
)

# -----------------------
# --- 4.3 Full output ---
# -----------------------

summary(
  fit_cfa_t0,
  fit.measures = TRUE,
  standardized = TRUE,
  rsquare      = TRUE
)

# Also print 95% confidence intervals for standardized factor loadings
std_loadings <- lavaan::standardizedSolution(fit_cfa_t0, ci = TRUE) |>
  subset(op == "=~") |>
  dplyr::select(lhs, rhs, est.std, ci.lower, ci.upper)

print(std_loadings)

# 5. Cross-Version Comparison: Dutch Original vs German Translated PPBS

## 5.1 Descriptive Cross-Version Comparison

Aim of this section:
- Prepare the original Dutch PPBS dataset assessed 8 weeks postpartum
- Bring the German T0 dataset into the same structure
- Label both versions clearly
- Create a combined dataset for later analyses
- Print version-specific descriptive statistics

In [ ]:
# --------------------------------------
# --- 1. Helper: labelled -> numeric ---
# --------------------------------------

to_num_labelled <- function(x) {
  if (inherits(x, "haven_labelled")) return(as.numeric(x))
  if (is.numeric(x)) return(x)
  suppressWarnings(as.numeric(trimws(as.character(x))))
}

# ----------------------------------------------
# --- 2. Dutch original PPBS (8w postpartum) ---
# ----------------------------------------------

# Select relevant variables from the original dataset
dut_ppbs_8w <- original_data[, c(
  "respnr",
  "ppbstot8wpp",
  "PPBS_1_8wPP_r",
  "PPBS_2_8wPP_r",
  "PPBS_3_8wPP_r",
  "PPBS_4_8wPP_r",
  "PPBS_5_8wPP_r"
)]

# Convert labelled variables to numeric
dut_ppbs_8w[] <- lapply(dut_ppbs_8w, to_num_labelled)

# Harmonise variable names
names(dut_ppbs_8w) <- c("id", "ppbs_sum", paste0("PPBS", 1:5))

# Check sum score as a safety check
dut_ppbs_8w$sum_check <- rowSums(dut_ppbs_8w[, paste0("PPBS", 1:5)], na.rm = FALSE)
cat("\nCHECK: Dutch PPBS sum score equals row sum of the five Dutch PPBS items\n")
print(table(dut_ppbs_8w$ppbs_sum == dut_ppbs_8w$sum_check, useNA = "ifany"))

# Remove helper variable
dut_ppbs_8w$sum_check <- NULL

# Add version label
dut_ppbs_8w$version <- "Dutch"

# ------------------------------------
# --- 3. Prepare German PPBS (T0) ---
# ------------------------------------

#-----------------------------------
# Preliminary check of comparability
#-----------------------------------

# Prepare date variables for the check
german_t0_timing <- data[, c(
  "id_redcap",
  "t0_assessments_complete",
  "d_datgeburt_k1",
  "date_t0"
), drop = FALSE]

# Cases with completed T0 only
german_t0_timing <- german_t0_timing[
  to_num(german_t0_timing$t0_assessments_complete) == 2,
]

# Convert dates to Date format
german_t0_timing$birth_date <- as.Date(german_t0_timing$d_datgeburt_k1)
german_t0_timing$t0_date    <- as.Date(german_t0_timing$date_t0)

# Calculate age at T0 in days and weeks
german_t0_timing$days_postpartum  <- as.numeric(german_t0_timing$t0_date - german_t0_timing$birth_date)
german_t0_timing$weeks_postpartum <- german_t0_timing$days_postpartum / 7

# Exclude implausible or missing values
german_t0_timing <- german_t0_timing[
  !is.na(german_t0_timing$weeks_postpartum) &
    german_t0_timing$weeks_postpartum >= 0,
]

# Summary table
german_t0_timing_summary <- data.frame(
  n = nrow(german_t0_timing),
  mean_weeks = round(mean(german_t0_timing$weeks_postpartum), 1),
  sd_weeks = round(sd(german_t0_timing$weeks_postpartum), 1),
  median_weeks = round(median(german_t0_timing$weeks_postpartum), 1),
  min_weeks = round(min(german_t0_timing$weeks_postpartum), 1),
  max_weeks = round(max(german_t0_timing$weeks_postpartum), 1)
)

cat("\nDESCRIPTIVE CHECK: German sample timing at baseline (T0)\n")
cat("Weeks postpartum at T0, calculated from birth date and T0 assessment date\n")
View(german_t0_timing_summary)

# Frequency table in whole weeks
german_t0_timing$weeks_postpartum_rounded <- floor(german_t0_timing$weeks_postpartum)

german_t0_timing_freq <- as.data.frame(
  table(german_t0_timing$weeks_postpartum_rounded)
)

names(german_t0_timing_freq) <- c("weeks_postpartum", "n")
german_t0_timing_freq$percent <- round(
  100 * german_t0_timing_freq$n / sum(german_t0_timing_freq$n), 1
)

cat("\nFREQUENCY TABLE: German sample timing at baseline (T0)\n")
cat("Distribution of weeks postpartum at T0, rounded down to whole weeks\n")
View(german_t0_timing_freq)

# Histogram
hist(
  german_t0_timing$weeks_postpartum,
  breaks = "Sturges",
  main = "German sample: weeks postpartum at T0",
  xlab = "Weeks postpartum at baseline (T0)"
)

#-----------------------
# Prepare German dataset
#-----------------------

# German comparison dataset based on complete T0 item-level cases
# This is the same item-level dataset used for the T0 CFA.
# Cases with prorated REDCap PPBS total scores but incomplete item data are excluded.
ger_ppbs_t0 <- items_t0

# Add sum score
ger_ppbs_t0$ppbs_sum <- rowSums(ger_ppbs_t0[, paste0("PPBS", 1:5)], na.rm = FALSE)

# Add version label
ger_ppbs_t0$version <- "German"

# ------------------------------------------------------
# --- 4. Bring both datasets into the same structure ---
# ------------------------------------------------------

# Keep only variables relevant for the comparison
dut_ppbs_8w <- dut_ppbs_8w[, c(paste0("PPBS", 1:5), "ppbs_sum", "version")]
ger_ppbs_t0 <- ger_ppbs_t0[, c(paste0("PPBS", 1:5), "ppbs_sum", "version")]

# ----------------------------------
# --- 5. Create combined dataset ---
# ----------------------------------

ppbs_cross <- rbind(dut_ppbs_8w, ger_ppbs_t0)

# Set version as a factor with a specific order
ppbs_cross$version <- factor(ppbs_cross$version, levels = c("Dutch", "German"))

# ----------------------------------------------
# --- 6. Print version-specific descriptives ---
# ----------------------------------------------

ppbs_cross_desc <- ppbs_cross %>%
  group_by(version) %>%
  summarise(
    n = dplyr::n(),
    across(
      .cols = c(paste0("PPBS", 1:5), "ppbs_sum"),
      .fns  = list(
        mean = ~ mean(.x, na.rm = TRUE),
        sd   = ~ sd(.x, na.rm = TRUE)
      ),
      .names = "{.col}_{.fn}"
    )
  )

cat("\nDESCRIPTIVE TABLE: Dutch and German PPBS item and sum-score descriptives\n")
cat("Dutch = original Dutch validation dataset at 8 weeks postpartum; German = WELCOME T0 complete item-level cases.\n")
cat("The table is displayed in wide format to mirror the underlying summary object.\n\n")

html_tab <- knitr::kable(
  ppbs_cross_desc,
  format = "html",
  digits = 3,
  caption = "Dutch and German PPBS item and sum-score descriptives"
)
IRdisplay::display_html(
  paste0(
    "<div style='overflow-x:auto; width:100%; font-size:11px;'>",
    html_tab,
    "</div>"
  )
)

# -----------------------------------------------------
# --- 7. Response distributions per item by version ---
# -----------------------------------------------------

cat("\nRESPONSE DISTRIBUTIONS: PPBS item response categories by version\n")
cat("Percentages are column percentages within each version.\n")

for (i in 1:5) {
  item <- paste0("PPBS", i)
  cat("\n", item, "\n")
  print(round(prop.table(table(ppbs_cross[[item]], ppbs_cross$version), margin = 2) * 100, 1))
}

# ---------------------------------------------------------------
# --- 8. Dutch vs German item response patterns: stacked bars ---
# ---------------------------------------------------------------

# Long format for the five items
ppbs_cross_long <- ppbs_cross %>%
  dplyr::select(version, PPBS1, PPBS2, PPBS3, PPBS4, PPBS5) %>%
  tidyr::pivot_longer(
    cols = starts_with("PPBS"),
    names_to = "item",
    values_to = "response"
  ) %>%
  dplyr::filter(!is.na(response))

# Collapse response categories: 0 and 1 -> 0/1, 2 -> 2, 3 -> 3
ppbs_cross_long <- ppbs_cross_long %>%
  dplyr::mutate(
    response_collapsed = dplyr::case_when(
      response %in% c(0, 1) ~ "0/1",
      response == 2 ~ "2",
      response == 3 ~ "3",
      TRUE ~ NA_character_
    )
  )

# Factor with fixed order
ppbs_cross_long$response_collapsed <- factor(
  ppbs_cross_long$response_collapsed,
  levels = c("0/1", "2", "3"),
  labels = c("0/1 (not at all / hardly)", "2 (much)", "3 (very much)")
)

# Percent per Item and Version
ppbs_cross_plot <- ppbs_cross_long %>%
  dplyr::count(item, version, response_collapsed) %>%
  dplyr::group_by(item, version) %>%
  dplyr::mutate(
    total_n = sum(n),
    percent = 100 * n / total_n
  ) %>%
  dplyr::ungroup()

In [ ]:
# Plot: HORIZONTAL

cat("\nFIGURE: Stacked bar plot of Dutch and German PPBS item response patterns\n")
cat("Response categories 0 and 1 are collapsed for visual comparability.\n")

# Percent-labels
ppbs_cross_plot <- ppbs_cross_plot %>%
  dplyr::mutate(
    label = ifelse(
      percent >= 1,
      paste0(round(percent, 1), "%"),
      ""
    )
  )

ppbs_cross_plot$item <- factor(
  ppbs_cross_plot$item,
  levels = c("PPBS1", "PPBS2", "PPBS3", "PPBS4", "PPBS5")
)

ggplot(ppbs_cross_plot, aes(x = version, y = percent, fill = response_collapsed)) +
  geom_col(width = 0.7) +
  geom_text(
    aes(label = label),
    position = position_stack(vjust = 0.5),
    size = 3,
    lineheight = 0.9
  ) +
  facet_wrap(~ item, ncol = 1) +
  coord_flip() +
  scale_x_discrete(labels = c("Dutch" = "Dutch", "German" = "German")) +
  scale_y_continuous(
    limits = c(0, 100),
    breaks = seq(0, 100, by = 25),
    labels = function(x) paste0(x, "%")
  ) +
  scale_fill_grey(start = 0.85, end = 0.35) +
  labs(
    title = "Item response patterns of the Dutch and German PPBS versions",
    x = "Version",
    y = "Percentage",
    fill = "Response category"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold"),
    panel.grid.minor = element_blank(),
    strip.text = element_text(face = "bold")
  ) 

## 5.2 Preliminary Cross-Version Measurement Invariance 

Aim of this section:
- Explore whether the German translation and the Dutch original PPBS version show a comparable measurement structure
- Comparison between:
  * Dutch  = original dataset, 8 weeks postpartum
  * German = German version, T0

Invariance levels following Svetina et al. / Wu & Estabrook:
- A) Baseline / configural model:
    * Same basic one-factor structure across groups.
    * Factor loadings and thresholds are freely estimated across groups.
- B) Threshold invariance:
    * Item thresholds are constrained to be equal across groups.
    * This is the ordinal analogue of testing equality in response-category thresholds.
- C) Threshold-and-loading invariance:
    * Both item thresholds and factor loadings are constrained to be equal across groups.
    * This corresponds to the more restrictive ordinal invariance model tested in the article.

This analysis should be interpreted preliminary because the groups differ not only in language but also in sample composition and assessment timing.

NOTE: categories 0/1 were collapsed because category 0 was empty for PPBS1 and PPBS5 in the German version

In [ ]:
# ---------------
# Prepare dataset
# ---------------

# Create a copy of the combined dataset
ppbs_cross_mi <- ppbs_cross

# Collapse categories 0 and 1
for (i in 1:5) {
  item <- paste0("PPBS", i)
  
  ppbs_cross_mi[[item]] <- ifelse(ppbs_cross_mi[[item]] <= 1, 1, ppbs_cross_mi[[item]])
  
  ppbs_cross_mi[[item]] <- factor(
    ppbs_cross_mi[[item]],
    levels = c(1, 2, 3),
    labels = c("0/1", "2", "3"),
    ordered = TRUE
  )
}

# Check cell counts again after recoding
cat("CHECK: Cell counts for cross-version measurement invariance\n")
cat("Response categories 0 and 1 were collapsed to 0/1.\n")
cat("These counts are checked before fitting the ordinal multi-group CFA models.\n")
for (i in 1:5) {
  item <- paste0("PPBS", i)
  cat("\n", item, "\n")
  print(table(Response = ppbs_cross_mi[[item]], Version = ppbs_cross_mi$version))
}

# ------------------------------------------------------------
# Article-based ordinal measurement invariance approach
# following Svetina et al. / Wu & Estabrook identification
# ------------------------------------------------------------

items_mi <- paste0("PPBS", 1:5)

ppbs_model <- 'Bonding =~ PPBS1 + PPBS2 + PPBS3 + PPBS4 + PPBS5'

# ------------------------------
# A) Baseline / configural model
# ------------------------------

syntax_baseline <- semTools::measEq.syntax(
  configural.model = ppbs_model,
  data             = ppbs_cross_mi,
  ordered          = items_mi,
  parameterization = "delta",
  ID.fac           = "std.lv",
  ID.cat           = "Wu.Estabrook.2016",
  group            = "version",
  group.equal      = "" # no equality constraints: configural/baseline model
)

fit_cross_baseline <- lavaan::cfa(
  model            = as.character(syntax_baseline),
  data             = ppbs_cross_mi,
  group            = "version",
  ordered          = items_mi,
  estimator        = "WLSMV",
  parameterization = "delta"
)

cat("\nMODEL A: Baseline / configural model\n")
cat("No equality constraints across Dutch and German versions.\n")
cat("Factor loadings and thresholds are freely estimated across groups.\n\n")
summary(fit_cross_baseline, fit.measures = TRUE, standardized = TRUE)

# -----------------------------
# B) Threshold invariance model
# -----------------------------

syntax_threshold <- semTools::measEq.syntax(
  configural.model = ppbs_model,
  data             = ppbs_cross_mi,
  ordered          = items_mi,
  parameterization = "delta",
  ID.fac           = "std.lv",
  ID.cat           = "Wu.Estabrook.2016",
  group            = "version",
  group.equal      = c("thresholds")
)

fit_cross_threshold <- lavaan::cfa(
  model            = as.character(syntax_threshold),
  data             = ppbs_cross_mi,
  group            = "version",
  ordered          = items_mi,
  estimator        = "WLSMV",
  parameterization = "delta"
)

cat("\nMODEL B: Threshold invariance model\n")
cat("Item thresholds are constrained equal across Dutch and German versions.\n\n")
summary(fit_cross_threshold, fit.measures = TRUE, standardized = TRUE)

# ---------------------------------------
# C) Threshold + loading invariance model
# ---------------------------------------

syntax_threshold_loading <- semTools::measEq.syntax(
  configural.model = ppbs_model,
  data             = ppbs_cross_mi,
  ordered          = items_mi,
  parameterization = "delta",
  ID.fac           = "std.lv",
  ID.cat           = "Wu.Estabrook.2016",
  group            = "version",
  group.equal      = c("thresholds", "loadings")
)

fit_cross_threshold_loading <- lavaan::cfa(
  model            = as.character(syntax_threshold_loading),
  data             = ppbs_cross_mi,
  group            = "version",
  ordered          = items_mi,
  estimator        = "WLSMV",
  parameterization = "delta"
)

cat("\nMODEL C: Threshold-and-loading invariance model\n")
cat("Item thresholds and factor loadings are constrained equal across versions.\n\n")
summary(fit_cross_threshold_loading, fit.measures = TRUE, standardized = TRUE)

# -------------------
# Extract fit indices
# -------------------

extract_fit_article <- function(fit_baseline, fit_threshold, fit_threshold_loading) {
  
  fit_tab <- tibble::tibble(
    model = c("baseline", "thresholds", "thresholds_loadings"),
    
    chisq_scaled = c(
      lavaan::fitMeasures(fit_baseline, "chisq.scaled"),
      lavaan::fitMeasures(fit_threshold, "chisq.scaled"),
      lavaan::fitMeasures(fit_threshold_loading, "chisq.scaled")
    ),
    df_scaled = c(
      lavaan::fitMeasures(fit_baseline, "df.scaled"),
      lavaan::fitMeasures(fit_threshold, "df.scaled"),
      lavaan::fitMeasures(fit_threshold_loading, "df.scaled")
    ),
    pvalue_scaled = c(
      lavaan::fitMeasures(fit_baseline, "pvalue.scaled"),
      lavaan::fitMeasures(fit_threshold, "pvalue.scaled"),
      lavaan::fitMeasures(fit_threshold_loading, "pvalue.scaled")
    ),
    rmsea_scaled = c(
      lavaan::fitMeasures(fit_baseline, "rmsea.scaled"),
      lavaan::fitMeasures(fit_threshold, "rmsea.scaled"),
      lavaan::fitMeasures(fit_threshold_loading, "rmsea.scaled")
    ),
    cfi_scaled = c(
      lavaan::fitMeasures(fit_baseline, "cfi.scaled"),
      lavaan::fitMeasures(fit_threshold, "cfi.scaled"),
      lavaan::fitMeasures(fit_threshold_loading, "cfi.scaled")
    ),
    tli_scaled = c(
      lavaan::fitMeasures(fit_baseline, "tli.scaled"),
      lavaan::fitMeasures(fit_threshold, "tli.scaled"),
      lavaan::fitMeasures(fit_threshold_loading, "tli.scaled")
    ),
    srmr = c(
      lavaan::fitMeasures(fit_baseline, "srmr"),
      lavaan::fitMeasures(fit_threshold, "srmr"),
      lavaan::fitMeasures(fit_threshold_loading, "srmr")
    )
  )
  
  fit_tab <- fit_tab %>%
    dplyr::mutate(
      delta_cfi_scaled   = c(NA, diff(cfi_scaled)),
      delta_rmsea_scaled = c(NA, diff(rmsea_scaled)),
      delta_srmr         = c(NA, diff(srmr))
    )
  
  fit_tab
}

# Cutoff criteria for changes in approximate fit indices
# Chen (2007) suggested stricter criteria under unequal sample sizes.
cutoff_delta_cfi   <- -0.005
cutoff_delta_rmsea <-  0.010

fit_cross_article_tab <- extract_fit_article(
  fit_cross_baseline,
  fit_cross_threshold,
  fit_cross_threshold_loading
)

fit_cross_article_tab <- fit_cross_article_tab %>%
  dplyr::mutate(
    chen_criteria_met = dplyr::case_when(
      is.na(delta_cfi_scaled) ~ NA,
      delta_cfi_scaled >= cutoff_delta_cfi &
        delta_rmsea_scaled <= cutoff_delta_rmsea ~ TRUE,
      TRUE ~ FALSE
    )
  )

cat("\nSUMMARY TABLE: Cross-version measurement invariance fit indices\n")
cat("Δ values refer to the preceding model.\n")
cat("Chen criteria are evaluated using ΔCFI >= -0.005 and ΔRMSEA <= 0.010.\n\n")

fit_cross_article_display <- fit_cross_article_tab %>%
  dplyr::select(
    model,
    chisq_scaled, df_scaled, pvalue_scaled,
    cfi_scaled, tli_scaled, rmsea_scaled, srmr,
    delta_cfi_scaled, delta_rmsea_scaled, delta_srmr,
    chen_criteria_met
  )
display_table(
  fit_cross_article_display,
  caption = "Cross-version measurement invariance fit indices",
  digits = 3
)

# ----------------
# Model comparison
# ----------------

# Baseline vs. threshold invariance
# Note: In the present data, the baseline and threshold models have the same degrees of freedom and identical fit indices. 
# Therefore, this comparison is not informative and should not be interpreted as a meaningful chi-square difference test.

cat("\nMODEL COMPARISON: Baseline/configural vs. threshold model\n")
cat("Note: This comparison is not informative if both models have identical df and fit.\n\n")
lavaan::lavTestLRT(fit_cross_baseline, fit_cross_threshold)

# Threshold invariance vs. threshold + loading invariance
# This comparison evaluates whether adding equality constraints on the factor loadings worsens model fit.

cat("\nMODEL COMPARISON: Threshold vs. threshold-and-loading model\n")
cat("This tests whether adding equality constraints on factor loadings worsens model fit.\n\n")
lavaan::lavTestLRT(fit_cross_threshold, fit_cross_threshold_loading)

# 6. Reliability

Reliability estimation: 
- Cronbach's Alpha: Estimates reliability under the assumption of essential tau-equivalence (assumes that each item loads equally on the factor; if this assumption is violated, alpha may underestimate reliability) 
- McDonald's Omega: Is based on the congeneric model and allows different loadings, making it more appropriate here
- For T1-T3, only control-group cases are used because later time points in the intervention group may be affected by the intervention

In [ ]:
calculate_ppbs_reliability_detailed <- function(df) {
  
  # Empty list for the later summary
  summary_list <- list()
  
  for (tp in names(ppbs_item_cols_by_time)) {
    
    cat("\n------------------------------------------------\n")
    cat("ANALYSIS FOR TIME POINT:", tp)
    cat("\n------------------------------------------------\n")
    
    comp_col  <- ppbs_complete_col_by_time[[tp]]
    item_cols <- ppbs_item_cols_by_time[[tp]]
    
    # T0: all complete cases
    # T1-T3: control group only and complete cases
    if (tp == "T0") {
      dft <- df[df[[comp_col]] == 2, item_cols, drop = FALSE]
    } else {
      dft <- df[df[[comp_col]] == 2 & df$gruppe == "Control", item_cols, drop = FALSE]
    }
    
    # Listwise Deletion
    dft <- na.omit(dft)
    colnames(dft) <- paste0("PPBS", 1:5)
    n_obs <- nrow(dft)
    
    cat("Number of complete cases:", n_obs, "\n")
    
    # --- 1. Cronbach's Alpha ---
    cat("\n--- CRONBACH'S ALPHA ---\n")
    a_res <- psych::alpha(dft, check.keys = FALSE)
    print(a_res)
    
    # --- 2. McDonald's Omega ---
    cat("\n--- MCDONALD'S OMEGA ---\n")
    o_res <- psych::omega(dft, nfactors = 1, poly = TRUE, plot = FALSE)
    print(o_res)
    
    # --- 3. Save summary ---
    summary_list[[tp]] <- data.frame(
      Timepoint = tp,
      N         = n_obs,
      Alpha     = round(a_res$total$raw_alpha, 3),
      Omega_t   = round(o_res$omega.tot, 3)
    )
  }
  
  # Create summary table
  final_summary <- do.call(rbind, summary_list)
  return(final_summary)
}

# Run
ppbs_reli_summary <- calculate_ppbs_reliability_detailed(ppbs_df)

# Print summary table
print(ppbs_reli_summary)

# 7. Evidence Based on Relations to Other Variables

## 7.1 Nomological Validity & Discriminant Validity

Aim of this section: Test expected negative associations between PPBS and psychological distress at T0

- T0 is used because this time point is not affected by the intervention
- DASS subscales: depression, anxiety, stress
- Spearman correlation: appropriate for ordinal data & robust to non-normality

Discriminant validity is examined through the pattern of associations:
- the association with depression is expected to be the strongest
- associations with anxiety and stress are expected to be weaker

In [ ]:
# ----------------------------------------------
# --- 1. T0 dataset for correlation analyses ---
# ----------------------------------------------

relations_t0 <- ppbs_dass_t0_df

# Keep only complete T0 cases on the relevant variables
relations_t0 <- na.omit(relations_t0[, c(
  "eltern_ppbs_score_t0",
  "eltern_dass_depression_t0",
  "eltern_dass_angst_t0",
  "eltern_dass_stress_t0"
)])

cat("Complete cases for PPBS-DASS correlations at T0:", nrow(relations_t0), "\n")

# ------------------------------------------
# --- 2. Calculate Spearman correlations ---
# ------------------------------------------

ppbs_dass_relations_t0 <- function(df) {
  
  outcomes <- c(
    depression = "eltern_dass_depression_t0",
    anxiety    = "eltern_dass_angst_t0",
    stress     = "eltern_dass_stress_t0"
  )
  
  res_list <- lapply(names(outcomes), function(outcome_name) {
    
    y_col <- outcomes[[outcome_name]]
    x <- df$eltern_ppbs_score_t0
    y <- df[[y_col]]
    
    ok <- is.finite(x) & is.finite(y)
    x <- x[ok]
    y <- y[ok]
    
    ct <- cor.test(x, y, method = "spearman", exact = FALSE)
    
    tibble::tibble(
      outcome = outcome_name,
      n       = length(x),
      r       = unname(ct$estimate),
      p_raw   = ct$p.value
    )
  })
  
  final_tab <- dplyr::bind_rows(res_list)
  
  final_tab <- final_tab %>%
    dplyr::mutate(
      p_adj = p.adjust(p_raw, method = "holm"),
      significant = ifelse(p_adj < 0.05, "*", "ns"),
      effect_size_category = dplyr::case_when(
        abs(r) >= 0.50 ~ "large",
        abs(r) >= 0.30 ~ "medium",
        abs(r) >= 0.10 ~ "small",
        TRUE           ~ "none"
      )
    )
  
  return(final_tab)
}

# Function call
tab_relations_t0 <- ppbs_dass_relations_t0(relations_t0)
View(tab_relations_t0)

# -----------------------------------------------
# --- 3. OPTIONAL: Scatterplot for depression ---
# -----------------------------------------------

# Note: because Spearman correlation reflects a monotonic but not necessarily linear association, the regression line is only a visual aid and should not be overinterpreted

dep_res <- tab_relations_t0[tab_relations_t0$outcome == "depression", ]

r_val <- round(dep_res$r, 2)
p_val <- round(dep_res$p_adj, 3)
sig_status <- dep_res$significant

plot_subtitle <- paste0(
  "Spearman's rho = ", r_val,
  " (p_adj = ", p_val, ", ", sig_status, ")"
)

ggplot(relations_t0, aes(x = eltern_ppbs_score_t0, y = eltern_dass_depression_t0)) +
  geom_jitter(
    width = 0.2,
    height = 0.2,
    alpha = 0.6,
    color = "steelblue"
  ) +
  geom_smooth(
    method = "lm",
    se = TRUE,
    color = "darkred",
    fill = "lightgray"
  ) +
  labs(
    title = "Association between Bonding and Depression (T0)",
    subtitle = plot_subtitle,
    x = "PPBS Total Score (Bonding)",
    y = "DASS Depression Score"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    axis.title = element_text(size = 12)
  )

## 7.2 Predictive Validity

Aim: To examine whether earlier PPBS scores (T0) predict later PPBS scores (T2)

The T2 assessment was used as the primary follow-up time point because the available sample size was larger than at T3.

NOTE: Predictive validity analyses were restricted to the control group because later PPBS scores in the intervention group may have been influenced by the intervention.

In [ ]:
# -------------------------------------------
# --- 1. Data preparation for T2 analysis ---
# -------------------------------------------

# Control group only + completed T0 and T2 assessments
t0_complete <- to_num(ppbs_df$t0_assessments_complete) == 2
t2_complete <- to_num(ppbs_df$t2_assessments_teil_1_kind_bonding_user_experience_complete) == 2

t0_complete[is.na(t0_complete)] <- FALSE
t2_complete[is.na(t2_complete)] <- FALSE

pred_data_t2 <- ppbs_df[
  ppbs_df$gruppe == "Control" &
    t0_complete &
    t2_complete,
  c("eltern_ppbs_score_t0", "eltern_ppbs_score_t2"),
  drop = FALSE
]

# Ensure numeric format
pred_data_t2[] <- lapply(pred_data_t2, to_num)

# Complete cases for regression
pred_data_t2 <- na.omit(pred_data_t2)

cat("Complete cases for predictive validity analysis T0 -> T2:", nrow(pred_data_t2), "\n")

# --------------------
# --- 2. Fit model ---
# --------------------

# Linear regression: PPBS T2 ~ PPBS T0
m_ppbs_t2 <- lm(eltern_ppbs_score_t2 ~ eltern_ppbs_score_t0, data = pred_data_t2)

# Model summary
summary(m_ppbs_t2)

# ----------------------------------
# --- 3. Check model assumptions ---
# ----------------------------------

# 1. Visual inspection
par(mfrow = c(2, 2))
plot(m_ppbs_t2, main = "Diagnostics: PPBS T2 ~ PPBS T0")
par(mfrow = c(1, 1))

# 2. Numerical check: Shapiro-Wilk test of residual normality
shapiro.test(residuals(m_ppbs_t2))

# 3. Numerical check: Breusch-Pagan test for homoskedasticity
lmtest::bptest(m_ppbs_t2)